# Simple RAG-LLM Action Planning

This notebook creates a RAG knowledge base from JSON files and generates action plans using LLM.

In [ ]:
# 1. Setup and Load Predictions
import os
import json
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# Paths
rag_knowledge_dir = Path("RAG_docs/knowledge")
rag_predictions_dir = Path("RAG_docs/predictions")
results_action_dir = Path("RAG_docs/action_plans")
os.makedirs(results_action_dir, exist_ok=True)

# Load prediction file (find latest)
prediction_files = list(rag_predictions_dir.glob("sample_predictions_detailed_*.json"))
if prediction_files:
    latest_prediction = max(prediction_files, key=lambda p: p.stat().st_mtime)
    with open(latest_prediction, "r", encoding="utf-8") as f:
        predictions_data = json.load(f)
    print(f"Loaded {len(predictions_data)} predictions from {latest_prediction.name}")
else:
    print("No prediction files found!")
    predictions_data = []


In [ ]:
# 2. Build RAG Knowledge Base from JSON Files
import json

rag_documents = []
rag_metadata = []

# Load all JSON files from knowledge folder
if rag_knowledge_dir.exists():
    json_files = list(rag_knowledge_dir.glob("*.json"))
    print(f"Found {len(json_files)} JSON files in knowledge folder")
    
    for json_file in json_files:
        print(f"  Loading: {json_file.name}")
        try:
            with open(json_file, "r", encoding="utf-8") as f:
                data = json.load(f)
            
            # Convert JSON to text chunks
            if isinstance(data, list):
                for idx, item in enumerate(data):
                    chunk_text = json.dumps(item, indent=2, ensure_ascii=False)
                    rag_documents.append(chunk_text)
                    rag_metadata.append({
                        "file": json_file.name,
                        "chunk": idx + 1,
                        "type": "JSON"
                    })
            elif isinstance(data, dict):
                # Split large dicts into chunks
                chunk_text = json.dumps(data, indent=2, ensure_ascii=False)
                # Simple chunking by size
                chunk_size = 2000
                for i in range(0, len(chunk_text), chunk_size):
                    chunk = chunk_text[i:i+chunk_size]
                    rag_documents.append(chunk)
                    rag_metadata.append({
                        "file": json_file.name,
                        "chunk": (i // chunk_size) + 1,
                        "type": "JSON"
                    })
        except Exception as e:
            print(f"    Error loading {json_file.name}: {e}")

print(f"\nTotal RAG documents loaded: {len(rag_documents)}")


In [ ]:
# 3. Simple RAG Search Function
def search_rag_knowledge(query, top_k=3):
    """Simple text-based search in RAG documents."""
    query_lower = query.lower()
    query_terms = query_lower.split()
    
    scored_docs = []
    for idx, doc in enumerate(rag_documents):
        doc_lower = doc.lower()
        score = sum(doc_lower.count(term) for term in query_terms)
        if score > 0:
            scored_docs.append({
                "content": doc[:500],  # First 500 chars
                "file": rag_metadata[idx].get("file", "unknown"),
                "chunk": rag_metadata[idx].get("chunk", 0),
                "score": score
            })
    
    # Sort by score and return top_k
    scored_docs.sort(key=lambda x: x["score"], reverse=True)
    return scored_docs[:top_k]

print("RAG search function ready")


In [ ]:
# 4. Create Simple Prompt and Process Sample
def create_simple_prompt(sample_data):
    """Create prompt from prediction details and RAG search."""
    # Get prediction details
    sample_id = sample_data.get("sample_id", 0)
    attack_type = sample_data.get("predicted_label", "UNKNOWN")
    confidence = sample_data.get("confidence", 0.0)
    
    # Get party contributions if available
    shap_explanation = sample_data.get("shap_explanation", {})
    party_contributions = shap_explanation.get("party_contributions_pct", {})
    
    # Search RAG knowledge base
    query = f"{attack_type} attack mitigation"
    rag_results = search_rag_knowledge(query, top_k=3)
    
    # Build RAG context
    rag_context = ""
    if rag_results:
        rag_context = "\n\nRAG Knowledge Base Context:\n"
        for idx, doc in enumerate(rag_results, 1):
            rag_context += f"\n[{idx}] {doc['file']} (Chunk {doc['chunk']}):\n{doc['content']}...\n"
    else:
        rag_context = "\n\nRAG Knowledge Base: No relevant documents found."
    
    # Build party contributions text
    party_text = ""
    if party_contributions:
        party_text = "\nParty Contributions:\n"
        for party, contrib in sorted(party_contributions.items(), key=lambda x: x[1], reverse=True)[:3]:
            party_text += f"  - {party}: {contrib:.1%}\n"
    
    # Create simple prompt
    prompt = f"""Analyze this attack detection and provide action plan.

PREDICTION DETAILS:
- Sample ID: {sample_id}
- Attack Type: {attack_type}
- Confidence: {confidence:.1%}
{party_text}
{rag_context}

Provide JSON response:
{{
  "threat_level": "Critical|High|Medium|Low",
  "primary_actions": ["Action 1", "Action 2", ...],
  "supporting_actions": ["Action 1", "Action 2", ...],
  "reasoning": "Brief explanation",
  "execution_priority": "Immediate|Fast-track|Standard|Monitor"
}}"""
    
    return prompt, rag_results

print("Prompt creation function ready")


In [ ]:
# 5. Call LLM API
def call_llm_api(prompt):
    """Call LLM API with simple prompt."""
    try:
        from openai import OpenAI
    except ImportError:
        print("ERROR: openai package not installed. Run: pip install openai")
        return None
    
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        print("ERROR: OPENAI_API_KEY not set in .env file")
        return None
    
    model = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
    temperature = float(os.getenv("OPENAI_TEMPERATURE", "0.3"))
    
    client = OpenAI(api_key=api_key)
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are a cybersecurity expert. Return only valid JSON."},
            {"role": "user", "content": prompt}
        ],
        temperature=temperature
    )
    
    # Parse response
    response_text = response.choices[0].message.content.strip()
    try:
        # Try to extract JSON
        start = response_text.find('{')
        end = response_text.rfind('}') + 1
        if start >= 0 and end > start:
            return json.loads(response_text[start:end])
    except:
        pass
    
    # Fallback
    return {
        "threat_level": "Unknown",
        "primary_actions": ["Unable to parse LLM response"],
        "supporting_actions": [],
        "reasoning": response_text[:200],
        "execution_priority": "Standard"
    }

# Process sample: Create prompt, search RAG, and call LLM
# Note: Currently processes first sample only. To process all, change to: for sample in predictions_data[:10]
if len(predictions_data) > 0:
    sample = predictions_data[0]  # Process first sample
    
    print("="*80)
    print("PROCESSING SAMPLE")
    print("="*80)
    print(f"Sample ID: {sample.get('sample_id')}")
    print(f"Attack Type: {sample.get('predicted_label')}")
    print(f"Confidence: {sample.get('confidence', 0):.1%}")
    
    # Create prompt with RAG search
    prompt, rag_results = create_simple_prompt(sample)
    print(f"\nFound {len(rag_results)} RAG documents")
    
    # Print complete RAG prompt
    print("\n" + "="*80)
    print("COMPLETE RAG PROMPT")
    print("="*80)
    print(prompt)
    print("="*80)
    
    # Call LLM
    print("\nCalling LLM API...")
    llm_response = call_llm_api(prompt)
    
    if llm_response:
        print("\n" + "="*80)
        print("LLM ACTION PLAN")
        print("="*80)
        print(f"Threat Level: {llm_response.get('threat_level', 'Unknown')}")
        print(f"Priority: {llm_response.get('execution_priority', 'Unknown')}")
        print(f"\nPrimary Actions:")
        for i, action in enumerate(llm_response.get('primary_actions', [])[:3], 1):
            print(f"  {i}. {action}")
        
        # Save result
        result = {
            "sample_id": sample.get("sample_id"),
            "prediction": {
                "predicted_label": sample.get("predicted_label"),
                "confidence": sample.get("confidence", 0.0)
            },
            "rag_documents_used": len(rag_results),
            "llm_action_plan": llm_response
        }
        
        output_file = results_action_dir / "simple_action_plan.json"
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(result, f, indent=2, ensure_ascii=False)
        
        print(f"\n✓ Saved to {output_file}")
